# Score por reglas sobre el dataset real

**Notebook de prueba (ML-2).** El pipeline vive en `xray.features`, `xray.labels`, `xray.rules`, `xray.explain` y `xray.evals` (spec: `docs/rules_spec.md`); aquí solo se importa y se mira. La §1 construye la tabla del contrato con `features.build()` (slice #2, hecho el 19 sep) y la guarda en `artifacts/features.parquet`, que es lo que leen `xray-evals` y `xray-score` por defecto.

Decisiones del builder en `docs/features_seam.md` §3: cuentas corrientes con saldo en `balances.csv`, mínimo sobre los días con movimiento, solo facturas de verdad (sin canceladas ni documentos de pago), corte en el último mes completo (2026-08).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from xray import features, labels, rules
from xray.data import load, artifacts_dir

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)

## 1. Tabla del contrato con `features.build()`

In [ ]:
feat = features.build()  # unos 10 s desde la caché parquet; features.validate() ya ha pasado
feat.to_parquet(artifacts_dir() / "features.parquet", index=False)
print(f"OK contrato · {len(feat):,} filas · {feat['company_id'].nunique()} empresas · {feat['month'].min()}…{feat['month'].max()}")
feat.describe().T.round(2)

## 2. Pipeline de reglas sobre la tabla provisional

`rules.run` ranquea dentro del mes, calcula el índice, el evento y la etiqueta, ajusta el mapa isotónico con `month ≤ 2025-08` y devuelve la tabla plana.

In [ ]:
cfg = rules.RulesConfig()
out = rules.run(feat, cfg=cfg, train_until="2025-08")
model = rules.fit(out, cfg, train_until="2025-08")
print(f"mapa isotónico: {len(model.knots_x)} nudos sobre {model.n_train:,} filas de train · lead_cutoff = {model.lead_cutoff:.1f}")
print(f"eventos: {out['event'].sum()} en {out.loc[out['event'], 'company_id'].nunique()} empresas (preview del notebook 01: 222 en 177 con umbrales absolutos)")
print(f"filas con score: {out['score'].notna().mean():.0%} · con etiqueta t+6: {out['label_t6'].notna().mean():.0%}")
pd.DataFrame({"outlook": out["outlook"].value_counts(normalize=True).round(3),
              "confidence": out["confidence"].value_counts(normalize=True).round(3)})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
grid_x = np.linspace(0, 1, 200)
axes[0].plot(grid_x, model.predict(grid_x)); axes[0].set_title("mapa isotónico nivel → score"); axes[0].set_xlabel("nivel_t"); axes[0].set_ylabel("score")
sc = out.dropna(subset=["score"])
q = sc.groupby("month")["score"].quantile([.1, .25, .5, .75, .9]).unstack()
axes[1].plot(q.index.astype(str), q[.5], color="k"); axes[1].fill_between(q.index.astype(str), q[.25], q[.75], alpha=.3); axes[1].fill_between(q.index.astype(str), q[.1], q[.9], alpha=.15)
axes[1].set_title("score por mes: p10–p90, p25–p75, mediana"); axes[1].tick_params(axis="x", rotation=90)
red_share = (out["n_red"] >= cfg.red_month_min).groupby(out["month"]).mean()
axes[2].plot(red_share.index, red_share.values); axes[2].set_title("% empresas en mes rojo (≥ 2 señales)"); axes[2].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()

## 3. Evals rápidas (versión de andar por casa; la definitiva va a `xray/evals.py`)

In [ ]:
TEST = [str(p) for p in pd.period_range("2025-09", "2026-02", freq="M")]
o = out.sort_values(["company_id", "month"]).copy()
g = o.groupby("company_id")["event"]
aucs = {}
for h in range(1, 13):
    fut = sum(g.shift(-k).fillna(False).astype(bool) for k in range(1, h + 1)).astype(bool)
    complete = g.shift(-h).notna()
    mask = o["month"].isin(TEST) & ~o["in_event"] & o["score"].notna() & complete
    y, s = fut[mask], -o.loc[mask, "score"]
    aucs[h] = roc_auc_score(y, s) if y.nunique() == 2 else np.nan
aucs = pd.Series(aucs, name="AUC(h) test")
print(aucs.round(3).to_string())

In [ ]:
red = (o["n_red"] >= cfg.red_month_min)
gr = red.groupby(o["company_id"])
base = red.mean()
pers = pd.Series({k: gr.shift(-k)[red].astype(float).mean() for k in range(1, 13)}, name="P(rojo t+k | rojo t)")
lift = (pers / base).rename("lift")
horizon = int(lift[lift >= 2].index.max()) if (lift >= 2).any() else 0
print(f"tasa base de mes rojo {base:.1%} · horizonte de persistencia (lift ≥ 2×): {horizon} meses")
pd.concat([pers.round(3), lift.round(2)], axis=1).T

In [ ]:
# direccionalidad: Δscore(t−3→t) frente a Δnivel(t→t+6), en test
gs = o.groupby("company_id")
d_score = o["score"] - gs["score"].shift(3)
d_level = gs["level"].shift(-6) - o["level"]
m = o["month"].isin(TEST) & d_score.notna() & d_level.notna()
print(f"Spearman Δscore(t−3→t) vs Δnivel(t→t+6): {d_score[m].corr(d_level[m], method='spearman'):.3f} sobre {m.sum():,} filas")
neg = o["outlook"].eq("negative") & m
stab = o["outlook"].eq("stable") & m
print(f"P(nivel baja a 6 m | outlook negativo) = {(d_level[neg] < 0).mean():.0%} · P(baja | estable) = {(d_level[stab] < 0).mean():.0%}")

## 4. Dos empresas: una con evento, una sin

In [ ]:
with_event = out.loc[out["event"] & out["month"].isin(TEST), "company_id"].value_counts().index[:1].tolist()
quiet = out.groupby("company_id").filter(lambda d: len(d) >= 18 and d["n_red"].max() == 0)["company_id"].unique()[:1].tolist()
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, cid in zip(axes, with_event + quiet):
    d = out[out["company_id"] == cid].sort_values("month")
    ax.plot(d["month"], d["score"], marker="o", label="score")
    ax.plot(d["month"], d["level"] * 100, ls="--", label="nivel × 100")
    for mth in d.loc[d["event"], "month"]:
        ax.axvline(mth, color="red", alpha=.5)
    ax.set_title(f"{cid} · outlook final: {d['outlook'].iloc[-1]} · confidence: {d['confidence'].iloc[-1]}")
    ax.tick_params(axis="x", rotation=90); ax.legend()
plt.tight_layout(); plt.show()
out[out["company_id"].isin(with_event)][["month", "n_red", "state_index", "level", "score", "outlook", "event", "confidence"]].tail(10)

## 5. Lo mismo sobre la fixture de tres empresas (lo que ven los tests)

In [ ]:
fx = rules.run(features.load_fixture(), events_ext=pd.read_csv(features.FIXTURE_PATH.parent / "events_mock.csv"))
fx.pivot(index="month", columns="company_id", values="score").round(1).join(
    fx.pivot(index="month", columns="company_id", values="outlook").add_prefix("outlook_"))